# Unit 2 Spring Refueling Outage — PERT Capability Demo

This notebook walks through the full PERT scheduling workflow applied to a realistic nuclear power plant (NPP) spring refueling outage.

**Scenario:** Unit 2 (pressurized water reactor) spring refueling outage (`UNIT2_RFO_SPRING`).  
The schedule comprises **66 work tasks** plus the synthetic `START` and `END` milestones (68 activities total).  
The target is to return the unit to power within a **17-day window** (408 h). The plant operates 24 h/day.

**Constraint features exercised in this case:**
- **Hold points** — T02 (QA), T11 (QA), T30 (NRC): work must pause for formal sign-off.
- **Discrete time windows** — T06 (eddy-current inspection has two allowed windows per Tech Spec).
- **Single time windows** — T15, T19, T28, T35 (Tech Spec surveillance deadlines).
- **System-state locks** — ECCS Train A/B enforced via `SystemStatePool` (OOS/OPERATIONAL states); SFP cooling trains mutually exclusive via `SFP_COOLING` state.
- **Consumables** — anti-contamination suits (`AC_SUIT`) deplete on start; a restock arrives at hour 96.
- **Dose budget** — RCT workers carry a 400 mRem/worker outage budget tracked as a consumable resource.
- **Multi-mode RCPSP** — T22 (SG-2 eddy-current inspection) has `normal` (24 h / 2 RCT / 1 rig) and `crash` (16 h / 3 RCT / 2 rigs) modes.
- **Mobilization lead** — T34 (electrical bus inspection) requires 12 h advance preparation.
- **Skill substitution** — T24 and T26 (RCP seal replacements) accept WELDER as an alternative to MECHANIC.
- **WBS groups** — `ECCS_A_MAINT`, `ECCS_B_MAINT`, `SG1_WORK`, `SG2_WORK` elevate group priority when any member's float reaches zero.
- **Multi-zone activity** — T28 (containment isolation valve test) simultaneously occupies `CONTAINMENT` and `ELEC_ROOM`.

In [ ]:
import sys
import logging

sys.path.insert(0, '../../../')

logging.basicConfig(level=logging.WARNING)  # suppress INFO chatter in demo

from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
from src.CPM.activity import Activity
import pandas as pd

print('Imports OK')

---
## Section 1 — Load and Inspect

`Pert.from_json_file()` validates the JSON against the schema, constructs the directed acyclic graph, builds all resource/equipment/location pools, and runs the initial CPM forward-backward pass.

Three diagnostic helpers are called below:
- `print_summary()` — activity count, resource pool sizes, equipment inventory.
- `debug_connectivity_and_es()` — verifies graph connectivity and reports early-start times with lag and mobilization-lead contributions.
- `debug_candidates_and_capacity(hours_ahead=48)` — shows which activities are schedulable in the first 48 h and the current capacity picture at each constraint dimension.

In [ ]:
pert = Pert.from_json_file(
    'npp_outage.json',
    schema_path='../outage_schema.json'
)

pert.print_summary()
print()
pert.debug_connectivity_and_es()
print()
pert.debug_candidates_and_capacity(hours_ahead=48)

---
## Section 2 — CPM Baseline (Resource-Unaware)

The **Critical Path Method** assumes unlimited resources: every activity starts as soon as its predecessors finish.  
This produces the **optimistic lower bound** on project duration — the shortest the outage could possibly be if resources were never a binding constraint.

For this case the CPM duration is **238.0 h (9.9 days)**, and the critical path runs through the reactor head removal/fuel-cycle spine:

> START → T01 → T02 → T03 → T04 → T05 → T06 → T07 → T08 → T09 → T10 → T11 → END

Note the 24-hour cooldown lag on T01→T02 and the 6-hour lag on T09→T10 — both baked into CPM early-start times.

In [ ]:
pert.generateInfo()

cpm_dur = pert.getProjectDuration()
cp = pert.getCriticalPath()

print(f'CPM duration : {cpm_dur:.1f} h  ({cpm_dur/24:.1f} days)')
print(f'Critical path: {" → ".join(a.name for a in cp)}')
print(f'Path length  : {len(cp)} activities')

---
## Section 3 — Resource-Constrained Scheduling: The Optimism Gap

CPM promises 238 h (9.9 days).  But the plant has real resource limits:
- 8 MECHANICs in early days, rising to 12 when contract crew arrives on day 5
- 5 ELECTRICIANs, 3 WELDERs throughout
- 6 RCT workers sharing a 400 mRem/worker dose budget
- One polar crane, one (later two) eddy-current rigs, one refueling machine
- Containment locked for the first 24 h; reactor cavity limited to one task at a time

**The optimism gap** is the difference between the CPM lower bound and the actual resource-constrained makespan.  
Its size depends critically on which **priority rule** is used to sequence competing activities.

### 3A — GRPW: what happens without smart scheduling

GRPW (Greatest Rank Position Weight) is a well-known heuristic that prioritises activities with the most downstream work.  It is a reasonable default but does not account for resource contention patterns.

In [ ]:
results_grpw = pert.calculateScheduleWithResources(
    sgs='max_use_res_ranked',
    priority_rule='grpw'
)

print('=== GRPW schedule ===')
print(f"  Makespan : {results_grpw['scheduled_duration']:.1f} h  "
      f"({results_grpw['scheduled_duration']/24:.1f} days)")
print(f"  CPM lower bound: {results_grpw['cpm_duration']:.1f} h")
ratio_grpw = results_grpw['scheduled_duration'] / results_grpw['cpm_duration']
print(f"  Ratio    : {ratio_grpw:.3f}  (1.000 = perfect)")
print(f"  Delay    : {results_grpw['delay_hours']:.1f} h  (accumulated activity wait time)")
print(f"  Completed: {results_grpw['n_completed']} / {results_grpw['n_activities']} activities")

### 3B — LF: optimized scheduling

**Latest Finish (LF)** prioritises activities closest to their CPM deadline.  
This naturally protects the critical path and minimises idle time between dependent tasks.

In [ ]:
results_lf = pert.calculateScheduleWithResources(
    sgs='max_use_res_ranked',
    priority_rule='lf'
)

print('=== LF schedule ===')
print(f"  Makespan : {results_lf['scheduled_duration']:.1f} h  "
      f"({results_lf['scheduled_duration']/24:.1f} days)")
print(f"  CPM lower bound: {results_lf['cpm_duration']:.1f} h")
ratio_lf = results_lf['scheduled_duration'] / results_lf['cpm_duration']
print(f"  Ratio    : {ratio_lf:.3f}  (1.000 = perfect)")
print(f"  Delay    : {results_lf['delay_hours']:.1f} h")
print(f"  Completed: {results_lf['n_completed']} / {results_lf['n_activities']} activities")

### 3C — Comparison

The two-row table below tells the core story:

- **GRPW produces 319 h (13.3 days)** — 34 % longer than the CPM lower bound.  That is 3.4 extra days of unplanned outage, costing the utility roughly \$1–2 M/day in replacement power.
- **LF recovers to 248 h (10.3 days)** — only 10 h (4.2 %) above the CPM lower bound.  The residual gap reflects irreducible resource contention on the critical chain; no greedy single-pass rule can eliminate it entirely.

This is the **core value proposition of PERT**: the right priority rule dramatically shrinks the optimism gap — from 81 h (GRPW) to 10 h (LF) — without adding resources or modifying the work scope.

In [ ]:
comparison = pd.DataFrame([
    {
        'Priority Rule': 'GRPW',
        'Makespan (h)': results_grpw['scheduled_duration'],
        'Days': round(results_grpw['scheduled_duration'] / 24, 1),
        'Ratio': round(ratio_grpw, 3),
        'Delay (h)': results_grpw['delay_hours'],
        'Completed': f"{results_grpw['n_completed']}/{results_grpw['n_activities']}",
    },
    {
        'Priority Rule': 'LF',
        'Makespan (h)': results_lf['scheduled_duration'],
        'Days': round(results_lf['scheduled_duration'] / 24, 1),
        'Ratio': round(ratio_lf, 3),
        'Delay (h)': results_lf['delay_hours'],
        'Completed': f"{results_lf['n_completed']}/{results_lf['n_activities']}",
    },
])
comparison.set_index('Priority Rule', inplace=True)
print('Priority rule comparison (CPM lower bound = 238.0 h)')
print(comparison.to_string())

---
## Section 4 — Schedule Quality: Fitness Score

`compute_fitness()` produces a composite scalar used as a GP/optimization training signal.  It aggregates four components:

| Component | Meaning | Weight |
|-----------|---------|--------|
| `makespan_ratio` | scheduled / CPM duration (ideal = 1.0) | α = 1.0 |
| `delay_ratio` | total wait hours / CPM duration (penalizes idle queuing) | β = 0.5 |
| `criticality_ratio` | fraction of real activities on resource-constrained critical chain (robustness proxy) | γ = 0.3 |
| `window_violation_ratio` | fraction of activities that missed a Tech Spec window | δ = 2.0 |

Lower composite is better.  Window violations receive the highest weight because missing a Technical Specification window is a regulatory failure.

In [ ]:
fitness = pert.compute_fitness()

fitness_display = pd.DataFrame([
    {'Component': 'makespan_ratio',         'Value': round(fitness['makespan_ratio'], 4),         'Weight (default)': 1.0},
    {'Component': 'delay_ratio',            'Value': round(fitness['delay_ratio'], 4),            'Weight (default)': 0.5},
    {'Component': 'criticality_ratio',      'Value': round(fitness['criticality_ratio'], 4),      'Weight (default)': 0.3},
    {'Component': 'window_violation_ratio', 'Value': round(fitness['window_violation_ratio'], 4), 'Weight (default)': 2.0},
    {'Component': '--- composite ---',      'Value': round(fitness['composite'], 4),              'Weight (default)': None},
])
fitness_display.set_index('Component', inplace=True)
print(fitness_display.to_string())
print()
print(f"Scheduled duration : {fitness['scheduled_duration']:.1f} h")
print(f"CPM duration       : {fitness['cpm_duration']:.1f} h")
print(f"Delay hours        : {fitness['delay_hours']:.1f} h")
print(f"Window violations  : {fitness['n_window_violations']}")

---
## Section 5 — Visualizations

All plots are written to interactive HTML files (Plotly) that can be opened in any browser.  
The LF schedule computed in Section 3 is the baseline for all visualizations.

### 5A — Gantt Chart

In [ ]:
# Opens as interactive HTML — view gantt_npp.html in browser
plot_gantt_chart(pert, filename='gantt_npp.html', show_delays=True)
print('Gantt chart written to gantt_npp.html')

### 5B — Activity DAG (Plotly)

The DAG view highlights both the CPM critical path (red) and the resource-constrained critical chain (orange).  
Augmented edges (resource wait arcs) are shown in grey — they represent the actual sequencing imposed by resource contention, not logical precedence.

In [ ]:
# Opens as interactive HTML — view dag_npp.html in browser
pert.plot_activity_dag(
    filename='dag_npp.html',
    library='plotly',
    highlight='both',
    layer_by='topo',
    include_augmented_edges=True,
    show_unscheduled=False
)
print('DAG written to dag_npp.html')

### 5C — Resource Utilization (all skills)

In [ ]:
for skill in pert.crew_pool.get_all_skills():
    fname = f'res_{skill}.html'
    plot_resource_utilization(pert, skill, filename=fname)
    print(f'  Resource utilization for {skill} written to {fname}')

### 5D — Location Utilization

In [ ]:
for loc in ['REACTOR_CAVITY', 'SG_BAY', 'CONTAINMENT']:
    fname = f'loc_{loc}.html'
    plot_location_utilization(pert, loc, filename=fname)
    print(f'  Location utilization for {loc} written to {fname}')

### 5E — Equipment Utilization

In [ ]:
for eq in pert.equipment_pool.get_all_equipment_ids():
    fname = f'eq_{eq}.html'
    plot_equipment_utilization(pert, eq, filename=fname)
    print(f'  Equipment utilization for {eq} written to {fname}')

---
## Section 6 — Schedule DataFrame

`get_schedule_dataframe()` returns one row per scheduled activity with columns:
- `activity_id`, `description`, `start_time`, `end_time`, `duration`
- `delay` — hours this activity waited for resources after its earliest start
- `on_resource_constrained_chain` — True if on the resource critical chain
- `tf_actual` — actual total float in the resource-constrained schedule

The sub-analyses below highlight activities with resource-driven waits and compare the CPM critical path membership against resource-constrained chain membership.

In [ ]:
df = pert.get_schedule_dataframe()
print(f'Schedule has {len(df)} rows')
print()
print(df.to_string(index=False))

In [ ]:
delayed = df[df['delay'] > 0].sort_values('delay', ascending=False)
print(f'Activities with resource-driven delay (n={len(delayed)}):')
cols = ['activity_id', 'description', 'delay', 'on_resource_constrained_chain']
print(delayed[[c for c in cols if c in delayed.columns]].to_string(index=False))

In [ ]:
cp_names = {a.name for a in pert.getCriticalPath()}
chain_col = 'on_resource_constrained_chain'

df['on_cpm_critical_path'] = df['activity_id'].isin(cp_names)
both  = df[df['on_cpm_critical_path'] & df[chain_col]]['activity_id'].tolist()
cpm_only = df[df['on_cpm_critical_path'] & ~df[chain_col]]['activity_id'].tolist()
chain_only = df[~df['on_cpm_critical_path'] & df[chain_col]]['activity_id'].tolist()

print(f'On both CPM critical path AND resource-constrained chain : {both}')
print(f'On CPM critical path only (freed by resources)          : {cpm_only}')
print(f'On resource-constrained chain only (resource bottleneck): {chain_only}')

---
## Section 7 — Augmented Graph: CPM Path vs Resource-Constrained Chain

The **augmented graph** extends the logical precedence DAG with **resource-wait arcs** — directed edges that record which activity was forced to wait because another held a shared resource.  The longest path through this augmented graph is the **resource-constrained critical chain**.

Key distinctions:
- **CPM total float** — computed from logical predecessors only; may be optimistic when resources are tight.
- **Actual total float** — computed in the augmented graph; accounts for resource waits; always ≥ 0.
- An activity may appear on the resource chain but not the CPM path (resource bottleneck pulled it critical).
- `explain_idle_on_chain()` identifies gaps on the chain where the critical resource was idle despite waiting tasks.

In [ ]:
import logging

_root = logging.getLogger()
_prev_level = _root.level
_root.setLevel(logging.DEBUG)

pert.print_chain_sets_summary()
print()
pert.explain_idle_on_chain()
print()
pert.explain_idle_on_chain_detailed()

_root.setLevel(_prev_level)

---
## Section 8 — Priority Rule Comparison

PERT ships 22 built-in priority rules.  They range from simple list-scheduling heuristics (LF, LS, EF, ES, duration) to composite rules (GRPW, GRD, MTS, MTP) and GP-derived rules (MEHH, GPHH).  
This sweep runs a representative subset and ranks them by makespan.

In [ ]:
rules_to_test = [
    'lf', 'ls', 'ef', 'es', 'grpw', 'grd', 'rr',
    'avgrr', 'maxrr', 'minrr', 'mts', 'mtp',
    'duration', 'random', 'irsm', 'wcs',
]

ok_results = {}
incomplete_results = {}
failed_rules = {}
for rule in rules_to_test:
    try:
        r = pert.calculateScheduleWithResources(
            sgs='max_use_res_ranked',
            priority_rule=rule
        )
        entry = {
            'makespan_h': r['scheduled_duration'],
            'days':       round(r['scheduled_duration'] / 24, 2),
            'ratio':      round(r['scheduled_duration'] / r['cpm_duration'], 3),
            'delay_h':    r['delay_hours'],
            'completed':  f"{r['n_completed']}/{r['n_activities']}",
        }
        if r['n_completed'] == r['n_activities']:
            ok_results[rule] = entry
        else:
            incomplete_results[rule] = entry
    except Exception as e:
        failed_rules[rule] = str(e)

df_rules = pd.DataFrame(ok_results).T.sort_values('makespan_h')
print('Priority rule comparison — complete schedules only (sorted by makespan):')
print(df_rules.to_string())

if incomplete_results:
    print()
    print(f'Rules with incomplete schedules ({len(incomplete_results)}) — makespan values are NOT comparable:')
    df_incomplete = pd.DataFrame(incomplete_results).T.sort_values('makespan_h')
    print(df_incomplete.to_string())
    print('  (Reported makespan = latest end among completed activities only; true project end is unknown.)')

if failed_rules:
    print()
    print(f'Rules that raised an exception ({len(failed_rules)}):')
    for rule, msg in failed_rules.items():
        print(f'  {rule}: {msg}')

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    # Only plot complete schedules — incomplete makespans are not valid project durations
    complete = df_rules[df_rules['makespan_h'].notna()].copy()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(complete.index, complete['makespan_h'], color='steelblue')
    ax.axvline(results_lf['cpm_duration'], color='red', linestyle='--', label='CPM lower bound (238 h)')
    ax.set_xlabel('Makespan (h)')
    ax.set_title('Makespan by Priority Rule — complete schedules (SGS = max_use_res_ranked)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('priority_rule_comparison.png', dpi=100)
    plt.close()
    print('Bar chart saved to priority_rule_comparison.png')
    if incomplete_results:
        print(f'Note: {list(incomplete_results.keys())} excluded (incomplete schedules).')
except ImportError:
    print('matplotlib not available — skipping bar chart')

---
## Section 9 — SGS Variant Comparison

Beyond the priority rule, the **Schedule Generation Scheme (SGS)** controls *how* the ranked candidate list is executed at each event step.  PERT provides two families:

**Parallel SGS** (`calculateScheduleWithResources`) — at each event, decide which subset of ready activities to start simultaneously.  Five variants control the selection policy:

| SGS | Description |
|-----|-------------|
| `first` | Serial-like: start only the single highest-priority *feasible* candidate; if resources are unavailable, wait rather than falling through to lower-priority candidates |
| `max_use_res_ranked` | Greedily pack as many candidates as resources allow, in ranked order |
| `max_use_res_shuffled` | Same but randomise order (diversification) |
| `md_knapsack` | Solve a mini knapsack to maximise resource utilisation at each step |
| `look_ahead` | Evaluate future opportunity cost before committing |

**Serial SGS** (`calculateSerialScheduleWithResources`) — process activities one at a time in a fixed priority order; for each activity, scan forward independently to find its earliest feasible start.  No global event clock; each activity "finds its own slot" in the already-committed profile.

**9A** sweeps the five parallel SGS variants.  **9B** sweeps the same priority rules through the serial engine.  A head-to-head comparison closes the section.

**Note on `first` vs `max_use_res_ranked`:** it may seem paradoxical that the serial-like SGS can outperform the greedy parallel one.  The reason is that `max_use_res_ranked` fills every idle resource slot at each step, which can schedule low-priority activities that consume the resources needed by a high-priority successor one step later.  `first` avoids that by keeping those resources idle until the top-priority activity is ready — effectively reserving capacity for the critical chain.

### 9A — Parallel SGS variants (priority\_rule = lf)

In [ ]:
sgs_variants = [
    'first',
    'max_use_res_ranked',
    'max_use_res_shuffled',
    'md_knapsack',
    'look_ahead',
]

sgs_results = {}
sgs_validation = {}

for sgs in sgs_variants:
    try:
        r = pert.calculateScheduleWithResources(sgs=sgs, priority_rule='lf')
        vr = pert.validate_schedule()
        sgs_results[sgs] = {
            'makespan_h': r['scheduled_duration'],
            'days':       round(r['scheduled_duration'] / 24, 2),
            'ratio':      round(r['scheduled_duration'] / r['cpm_duration'], 3),
            'delay_h':    r['delay_hours'],
            'completed':  f"{r['n_completed']}/{r['n_activities']}",
            'violations': len(vr.violations),
            'warnings':   len(vr.warnings),
            'feasible':   vr.is_feasible,
        }
        sgs_validation[sgs] = vr
    except Exception as e:
        sgs_results[sgs] = {
            'makespan_h': None, 'days': None, 'ratio': None,
            'delay_h': None, 'completed': 'ERROR', 'violations': None,
            'warnings': None, 'feasible': None,
        }

df_sgs = pd.DataFrame(sgs_results).T.sort_values('makespan_h')
print('SGS variant comparison (priority_rule=lf):')
print(df_sgs.to_string())

# Print full validation report for any infeasible schedule
for sgs, vr in sgs_validation.items():
    if not vr.is_feasible:
        print(f'\n=== Validation violations for sgs={sgs} ===')
        print(vr.summary())

### 9B — Serial SGS across priority rules

`calculateSerialScheduleWithResources` is a fundamentally different engine: it processes activities one at a time in a fixed priority order, and for each activity independently scans forward to find its earliest feasible start given the already-committed profile.  There is no global event clock and no notion of "what else can start now."

The same set of priority rules used in Section 8 is swept here so that the best serial result can be compared against the best parallel result.

In [ ]:
rules_to_test = [
    'lf', 'ls', 'ef', 'es', 'grpw', 'grd', 'rr',
    'avgrr', 'maxrr', 'minrr', 'mts', 'mtp',
    'duration', 'random', 'irsm', 'wcs',
]

serial_ok = {}
serial_incomplete = {}
serial_failed = {}
serial_validation = {}

for rule in rules_to_test:
    try:
        r = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        vr = pert.validate_schedule()
        entry = {
            'makespan_h': r['scheduled_duration'],
            'days':       round(r['scheduled_duration'] / 24, 2),
            'ratio':      round(r['scheduled_duration'] / r['cpm_duration'], 3),
            'delay_h':    r['delay_hours'],
            'completed':  f"{r['n_completed']}/{r['n_activities']}",
            'violations': len(vr.violations),
            'warnings':   len(vr.warnings),
            'feasible':   vr.is_feasible,
        }
        serial_validation[rule] = vr
        if r['n_completed'] == r['n_activities']:
            serial_ok[rule] = entry
        else:
            serial_incomplete[rule] = entry
    except Exception as e:
        serial_failed[rule] = str(e)

df_serial = pd.DataFrame(serial_ok).T.sort_values('makespan_h')
print('Serial SGS — complete schedules (sorted by makespan):')
print(df_serial.to_string())

if serial_incomplete:
    print()
    print(f'Serial SGS — incomplete schedules ({len(serial_incomplete)}) — makespan not comparable:')
    print(pd.DataFrame(serial_incomplete).T.sort_values('makespan_h').to_string())

if serial_failed:
    print()
    print(f'Serial SGS — exceptions ({len(serial_failed)}):')
    for rule, msg in serial_failed.items():
        print(f'  {rule}: {msg}')

# Print full validation report for any infeasible schedule
for rule, vr in serial_validation.items():
    if not vr.is_feasible:
        print(f'\n=== Validation violations for rule={rule} ===')
        print(vr.summary())

In [ ]:
# Head-to-head: best parallel SGS vs best serial SGS (complete + feasible schedules only)
cpm_h = pert.getProjectDuration()

feasible_parallel = df_sgs[df_sgs['feasible'] == True]
feasible_serial   = df_serial[df_serial['feasible'] == True]

best_parallel_rule = feasible_parallel['makespan_h'].idxmin()
best_parallel_h    = feasible_parallel.loc[best_parallel_rule, 'makespan_h']

best_serial_rule   = feasible_serial['makespan_h'].idxmin()
best_serial_h      = feasible_serial.loc[best_serial_rule, 'makespan_h']

print(f"CPM lower bound          : {cpm_h:.1f} h")
print()
print(f"Best parallel SGS        : {best_parallel_rule:20s}  {best_parallel_h:.1f} h  "
      f"(ratio {best_parallel_h/cpm_h:.3f})")
print(f"Best serial   SGS        : {best_serial_rule:20s}  {best_serial_h:.1f} h  "
      f"(ratio {best_serial_h/cpm_h:.3f})")
print()
gap = best_serial_h - best_parallel_h
print(f"Serial vs parallel gap   : {gap:+.1f} h  "
      f"({'serial better' if gap < 0 else 'parallel better' if gap > 0 else 'tied'})")

n_infeasible_par = int((df_sgs['feasible'] == False).sum())
n_infeasible_ser = int((df_serial['feasible'] == False).sum())
if n_infeasible_par or n_infeasible_ser:
    print()
    print(f"Infeasible schedules excluded from comparison: "
          f"{n_infeasible_par} parallel, {n_infeasible_ser} serial")

---
## Section 10 — Multi-Mode RCPSP

Task **T22** (Steam Generator 2 eddy-current tube inspection) is declared with two execution modes:

| Mode | Duration | RCT workers | EC rigs | Dose rate |
|------|----------|-------------|---------|----------|
| `normal` (default) | 24 h | 2 | 1 | 5 mRem/h |
| `crash` | 16 h | 3 | 2 | 6 mRem/h |

Crash mode saves 8 hours on T22 at the cost of one extra RCT worker and an extra EC rig.

Two questions are worth distinguishing:

1. **Does compressing T22 shorten the critical path?**  T22 has significant float in the normal schedule (~50 h), so it is not on the resource-constrained critical chain.  Even a perfect T22 compression would not reduce the project makespan at all.

2. **Can crash mode actually make the schedule *worse*?**  Yes — this is a classic multi-mode RCPSP pitfall.  Crash mode doubles the EC rig demand (1 → 2 rigs).  T20 (SG-1 eddy-current inspection) runs concurrently and already holds one rig; with only two rigs on site from day 2, crash-mode T22 cannot start alongside T20 and must wait for it to finish.  The 24 h wait more than cancels the 8 h duration saving, leaving T22 *later* than in normal mode.

In [ ]:
# Run a fresh LF schedule on a clean clone so Section 9's runs don't pollute the baseline
pert_normal = pert.clone_for_analysis()
r_normal = pert_normal.calculateScheduleWithResources(
    sgs='max_use_res_ranked', priority_rule='lf'
)

df_normal = pert_normal.get_schedule_dataframe()
t22_row = df_normal[df_normal['activity_id'] == 'T22']
print('T22 in normal-mode LF schedule:')
if len(t22_row):
    row = t22_row.iloc[0]
    print(f"  Start : {row['start_time']}")
    print(f"  End   : {row['end_time']}")
    print(f"  Delay : {row['delay']:.1f} h")
print(f"  Project makespan (normal): {r_normal['scheduled_duration']:.1f} h")

In [ ]:
pert_crash = pert.clone_for_analysis()
pert_crash.set_modes({'T22': 'crash'})
r_crash = pert_crash.calculateScheduleWithResources(
    sgs='max_use_res_ranked',
    priority_rule='lf'
)

df_crash = pert_crash.get_schedule_dataframe()

print('=== EC rig consumers (T20 and T22) ===')
print(f"{'':32s}  {'normal':>22s}  {'crash':>22s}")
for act_id in ['T20', 'T22']:
    rn = df_normal[df_normal['activity_id'] == act_id].iloc[0]
    rc = df_crash [df_crash ['activity_id'] == act_id].iloc[0]
    print(f"  {act_id} {rn['description'][:26]:28s}"
          f"  {rn['start_time'].strftime('%m-%d %H:%M')}→{rn['end_time'].strftime('%m-%d %H:%M')}"
          f"  {rc['start_time'].strftime('%m-%d %H:%M')}→{rc['end_time'].strftime('%m-%d %H:%M')}"
          f"  delay={rc['delay']:.0f}h")

print()
print(f"EC rigs available  : 1 rig until Apr 9 00:00, then 2 rigs")
print(f"T22 normal needs   : 1 rig  → can share with T20 (1 rig) from Apr 9")
print(f"T22 crash  needs   : 2 rigs → T20 holds 1, only 1 free → must wait for T20 to finish")
print()
delta = r_normal['scheduled_duration'] - r_crash['scheduled_duration']
print(f"Project makespan — normal: {r_normal['scheduled_duration']:.1f} h   "
      f"crash: {r_crash['scheduled_duration']:.1f} h   "
      f"delta: {delta:+.1f} h  ({'improvement' if delta > 0 else 'regression'})")

**Takeaway:** Crash mode is counterproductive here for two independent reasons:

- **Wrong activity** — T22 is off the resource-constrained critical chain (~50 h of float).  Compressing it cannot reduce the project makespan regardless of how it is scheduled.
- **Equipment contention** — doubling the EC rig demand forces T22 to serialize behind T20 instead of running in parallel.  The 24 h forced wait exceeds the 8 h duration saving, so crash-mode T22 finishes *later* than normal-mode T22.

The correct target for crash compression is any activity **on the constrained chain** with **slack in its shared resource pool** — identified in Sections 6–7 above.

---
## Section 11 — CCPM Buffers

**Critical Chain Project Management (CCPM)** replaces informal task padding with structured, explicitly sized buffers:
- **Project Buffer (PB)** — inserted at the end of the resource-constrained critical chain.  Absorbs disruptions anywhere on the chain so individual delays do not immediately extend the outage.
- **Feeding Buffers (FB)** — inserted where non-critical paths merge into the critical chain.  Protect the chain from merge-point delays.

Both are sized using the **Sum of Squares (SSQ)** method: $\text{buffer} = \text{fraction} \times \sqrt{\sum d_i^2}$ where $d_i$ are the durations of chain/path activities.  This is statistically grounded (combines variances in quadrature) and consistently smaller than the 50% cut rule.

In [ ]:
# Buffers require a fresh schedule run on a clean clone
pert_ccpm = pert.clone_for_analysis()
pert_ccpm.calculateScheduleWithResources(sgs='max_use_res_ranked', priority_rule='lf')

pert_ccpm.insert_project_buffer(method='ssq')
pert_ccpm.insert_feeding_buffers(method='ssq')

buf_status = pert_ccpm.get_buffer_status()

print('Buffer status:')
for bname, binfo in buf_status.items():
    print(f"  {bname}: size={binfo.get('size_hours', '?'):.1f} h  "
          f"type={binfo.get('buffer_type', '?')}  "
          f"consumed={binfo.get('consumed_hours', 0):.1f} h")

---
## Section 12 — Mid-Outage Replanning

At hour 72 (end of day 3) two disruptions occur simultaneously:
1. **Emergent work**: turbine vibration readings during coastdown trigger an unplanned inspection (`T_EMERG`: 3 h, 2 MECHANIC, TURBINE_HALL, must precede T33 reassembly).
2. **Resource loss**: 3 mechanics call in sick — available MECHANIC count drops from 12 to 5 effective h = 72.

`replan()` snapshots the schedule state at the trigger time, locks completed activities, adjusts pool availability, injects the new activity, and re-runs the scheduling loop over the remaining horizon.

In [ ]:
# Show original schedule state at h=72
from datetime import timedelta

t72 = pert.startTime + timedelta(hours=72)
df_snap = pert.get_schedule_dataframe().copy()

completed_at_72  = df_snap[df_snap['end_time']   <= t72]
in_progress_at_72 = df_snap[(df_snap['start_time'] <= t72) & (df_snap['end_time'] > t72)]
pending_at_72    = df_snap[df_snap['start_time']  >  t72]

print(f'Hour 72 snapshot ({t72})')
print(f'  Completed   : {len(completed_at_72)} activities — {list(completed_at_72["activity_id"])}')
print(f'  In progress : {len(in_progress_at_72)} activities — {list(in_progress_at_72["activity_id"])}')
print(f'  Pending     : {len(pending_at_72)} activities')

In [ ]:
pert_clone = pert.clone_for_analysis()
pert_clone.calculateScheduleWithResources(sgs='max_use_res_ranked', priority_rule='lf')

t_emerg = Activity(
    name='T_EMERG',
    duration=3.0,
    description='Turbine vibration inspection (emergent)',
    required_resources=[{'skill_type': 'MECHANIC', 'crew_count': 2}],
    location_id='TURBINE_HALL',
    childs=['T33']
)

result_replan = pert_clone.replan(
    current_time_hours=72.0,
    new_activities=[t_emerg],
    predecessor_wiring={'T_EMERG': ['T32']},
    resource_updates=[{'skill_type': 'MECHANIC', 'from_hour': 72, 'new_count': 5}],
    sgs='max_use_res_ranked'
)

print(f"Original makespan  : {results_lf['scheduled_duration']:.1f} h")
print(f"Replanned makespan : {result_replan['scheduled_duration']:.1f} h")
delta_replan = result_replan['scheduled_duration'] - results_lf['scheduled_duration']
print(f"Impact             : {delta_replan:+.1f} h")

violations, ok = pert_clone.check_dependency_violations()
print(f"Dependency violations: {len(violations)}  Feasible: {ok}")

In [ ]:
# Opens as interactive HTML — view gantt_replanned.html in browser
plot_gantt_chart(pert_clone, filename='gantt_replanned.html', show_delays=True)
print('Replanned Gantt chart written to gantt_replanned.html')

---
## Section 13 — Validation and Export

Final checks before handing off the schedule:
- `check_dependency_violations()` confirms all predecessor-finish ≤ successor-start relationships hold (with lag applied).
- The schedule DataFrame is exported to CSV for integration with outage management systems.

A clean schedule should show **0 dependency violations** and all 68 activities complete.

In [ ]:
# Re-run LF schedule on the original pert object for the final report
results_final = pert.calculateScheduleWithResources(sgs='max_use_res_ranked', priority_rule='lf')

violations_dep, feasible = pert.check_dependency_violations()
print(f'Dependency violations : {len(violations_dep)}')
print(f'Schedule feasible     : {feasible}')
print(f'Activities completed  : {results_final["n_completed"]} / {results_final["n_activities"]}')
print(f'Window violations     : {len(pert._window_violations)}')

if violations_dep:
    print()
    print('Violation details:')
    for v in violations_dep:
        print(f"  {v['predecessor']} → {v['successor']}: "
              f"pred_end={v['pred_end_time']}, succ_start={v['succ_start_time']}, "
              f"lag={v['lag_hours']:.1f}h, overlap={v['overlap_hours']:.2f}h")

print()

df_final = pert.get_schedule_dataframe()
df_final.to_csv('npp_schedule.csv', index=False)
print('Schedule exported to npp_schedule.csv')
print()
print('First 20 rows:')
print(df_final.head(20).to_string(index=False))